# 02 - Analisis Exploratorio de Datos (EDA)

## Objetivo
Explorar los datos generados para entender:
1. Distribuciones de las metricas de fibra optica
2. Correlaciones entre variables
3. Patrones temporales (diurnos, semanales)
4. Diferencias entre estados normales y de falla
5. Caracteristicas de las firmas pre-falla

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.visualization import (
    apply_style, plot_sensor_timeline, plot_distribution_by_fault,
    plot_correlation_matrix, FAULT_COLORS
)
from src.utils.config_loader import get_path

apply_style()

# Cargar datos
df = pd.read_parquet(get_path('raw_data') / 'fiber_readings.parquet')
print(f"Shape: {df.shape}")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nTipos de datos:")
print(df.dtypes)
df.head()

In [ ]:
# Resumen estadistico por tipo de falla
print("="*60)
print("ESTADISTICAS POR TIPO DE FALLA")
print("="*60)

metrics = ['optical_power_dbm', 'attenuation_db_km', 'ber', 'osnr_db', 'chromatic_dispersion']
for metric in metrics:
    print(f"\n--- {metric} ---")
    print(df.groupby('fault_type')[metric].describe()[['mean', 'std', 'min', 'max']])

In [ ]:
# Distribuciones por tipo de falla
for metric in metrics:
    fig = plot_distribution_by_fault(df, metric)
    plt.show()

In [ ]:
# Matriz de correlacion
fig = plot_correlation_matrix(df)
plt.show()

print("\nInterpretacion:")
print("- Potencia optica y SNR tienen alta correlacion positiva (ambas miden calidad de senal)")
print("- BER y SNR tienen correlacion negativa (mas ruido = mas errores)")
print("- Temperatura y humedad tienen correlacion negativa (patron diurno inverso)")

In [ ]:
# Patron diurno - promedio por hora
df['hour'] = df['timestamp'].dt.hour

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Patron Diurno - Promedio por Hora del Dia', fontsize=14, fontweight='bold')

for ax, metric in zip(axes.flat[:5], metrics):
    hourly = df[df['fault_type']=='normal'].groupby('hour')[metric].mean()
    ax.plot(hourly.index, hourly.values, 'o-', color='steelblue', linewidth=2)
    ax.set_xlabel('Hora del dia')
    ax.set_title(metric)
    ax.grid(True, alpha=0.3)

axes[1, 2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar segmentos con cada tipo de falla
for fault_type in ['physical_cut', 'degradation', 'bad_splice']:
    seg = df[df['fault_type'] == fault_type]['segment_id'].unique()[0]
    fig = plot_sensor_timeline(df, seg)
    plt.show()
    print(f"Segmento {seg}: {fault_type}\n")

In [ ]:
# Analisis de datos faltantes
print("Valores nulos por columna:")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.any() else "No hay valores nulos")

print(f"\nBalance de clases (fault_in_24h equivalente):")
pre_fault_or_fault = ((df['is_fault'] == 1) | ((df['pre_fault_hours'] > 0) & (df['pre_fault_hours'] <= 24)))
print(f"  Normal: {(~pre_fault_or_fault).sum():,}")
print(f"  Falla en 24h: {pre_fault_or_fault.sum():,}")
print(f"  Ratio: {pre_fault_or_fault.mean()*100:.1f}%")